In [1]:
# importações
import pandas as pd
import numpy as np

In [2]:
# Carregando base de dados

df = pd.read_csv(
    '../data/raw/202401_citibike_tripdata_1.csv',
    dtype={'start_station_id': str, 'end_station_id': str},
    parse_dates=['started_at', 'ended_at']
)

In [3]:
df2 = pd.read_csv(
    '../data/raw/202401_citibike_tripdata_2.csv',
    dtype={'start_station_id': str, 'end_station_id': str},
    parse_dates=['started_at', 'ended_at']
)

In [4]:
df2.columns


Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

In [5]:
df['started_at']

0        2024-01-01 13:00:04.563
1        2024-01-08 19:36:43.520
2        2024-01-12 15:00:41.580
3        2024-01-12 16:52:19.025
4        2024-01-05 19:50:19.202
                   ...          
999995   2024-01-19 08:11:31.812
999996   2024-01-23 08:50:57.782
999997   2024-01-21 14:16:24.984
999998   2024-01-23 09:02:49.848
999999   2024-01-26 09:39:15.238
Name: started_at, Length: 1000000, dtype: datetime64[us]

In [6]:
# Ajustando colunas temporais
COLS_TEMPORAIS = ['started_at', 'ended_at']

# Escalas das colunas ordinais (do menor para o maior nível)
ESCALA_QUALIDADE = []
ESCALA_ACABAMENTO_PORAO = []

ESCALAS_ORDINAIS = {}
COLS_ORDINAIS = list(ESCALAS_ORDINAIS)

# lista de dicionários para armazenar informações sobre as colunas
col_info = []

# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]

    # tipo e valores (menor/maior nível, ou primeiro/último para nominais)
    if col in COLS_TEMPORAIS or pd.api.types.is_datetime64_any_dtype(serie):
        col_type = 'Temporal'
        values = [serie.min(), serie.max()]
    elif col in COLS_ORDINAIS:
        col_type = 'Ordinal'
        escala = ESCALAS_ORDINAIS[col]
        if escala is None:
            values = [serie.min(), serie.max()]
        else:
            presentes = [nivel for nivel in escala if nivel in set(serie.dropna())]
            values = [presentes[0], presentes[-1]] if presentes else [None, None]
    elif serie.dtype == 'int64':
        col_type = 'Discreta'
        values = [serie.min(), serie.max()]
    elif serie.dtype == 'float64':
        col_type = 'Contínua'
        values = [serie.min(), serie.max()]
    else:  # object, str, category, bool
        col_type = 'Nominal'
        uniques = serie.dropna().unique()
        values = [uniques[0], uniques[-1]] if len(uniques) else [None, None]

    # descrição (vamos supor que você já tenha uma descrição)
    description = 'Descrição da coluna'

    # adicionando informações à lista de dicionários
    col_info.append({
        'variavel': col,
        'descricao': description,
        'valores': values,
        'Tipo': col_type
    })

# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

In [7]:
# Descrições de negócio, por nome da coluna em português
DESCRICOES = {
    "ride_id": "Identificador único alfanumérico de cada viagem",
    "rideable_type": "Tipo de bicicleta utilizada (ex: clássica, elétrica)",
    "started_at": "Data e hora exata do início da viagem",
    "ended_at": "Data e hora exata do término da viagem",
    "start_station_name": "Nome da estação onde a viagem foi iniciada",
    "start_station_id": "Código identificador da estação de origem",
    "end_station_name": "Nome da estação onde a viagem foi finalizada",
    "end_station_id": "Código identificador da estação de destino",
    "start_lat": "Latitude da estação ou ponto de partida",
    "start_lng": "Longitude da estação ou ponto de partida",
    "end_lat": "Latitude da estação ou ponto de chegada",
    "end_lng": "Longitude da estação ou ponto de chegada",
    "member_casual": "Tipo de usuário que realizou a viagem (membro anual ou cliente casual)"
}

# Adicionando descrição para cada uma das variáveis selecionadas
descricoes = [DESCRICOES[original] for original in col_info_df["variavel"]]

assert len(descricoes) == len(col_info_df)

# Adicionando descrição para cada uma das variáveis selecionadas
for i in range(0, len(col_info_df)):
    col_info_df.loc[i, "descricao"] = descricoes[i]

col_info_df

,variavel,descricao,valores,Tipo
0,ride_id,Identificador único alfanumérico de cada viagem,"[8E865410DBDE0CA9, C29B38D0D93E10BA]",Nominal
1,rideable_type,"Tipo de bicicleta utilizada (ex: clássica, elé...","[electric_bike, classic_bike]",Nominal
2,started_at,Data e hora exata do início da viagem,"[2023-12-31 02:36:55.648000, 2024-01-31 23:56:...",Temporal
3,ended_at,Data e hora exata do término da viagem,"[2024-01-01 00:00:08.272000, 2024-01-31 23:59:...",Temporal
4,start_station_name,Nome da estação onde a viagem foi iniciada,"[3 St & 3 Ave, 2 Ave & 37 St]",Nominal
5,start_station_id,Código identificador da estação de origem,"[4028.03, 7154.10]",Nominal
6,end_station_name,Nome da estação onde a viagem foi finalizada,"[Carroll St & Smith St, 6 St & Grand St]",Nominal
7,end_station_id,Código identificador da estação de destino,"[4225.14, HB302]",Nominal
8,start_lat,Latitude da estação ou ponto de partida,"[40.633385, 40.8863]",Contínua
9,start_lng,Longitude da estação ou ponto de partida,"[-74.0506564, -73.84978]",Contínua


In [8]:
# Salvando dicionário de dados
col_info_df["valores"] = col_info_df["valores"].apply(
    lambda valores: [valor.item() if isinstance(valor, np.generic) else valor for valor in valores]
)
col_info_df.to_csv('../data/processed/dicionario.csv',index=False,sep=';')

In [9]:
# Salvando escalas ordinais
# Escala usada para as ordinais numéricas (marcadas como None no dicionário)
'''
ESCALA_NUMERICA_PADRAO = list(range(1, 11))

# Convertendo o dicionário de escalas para o formato longo
linhas_escala = []
for coluna, niveis in ESCALAS_ORDINAIS.items():
    if niveis is None:
        niveis = ESCALA_NUMERICA_PADRAO
    for ordem, nivel in enumerate(niveis, start=1):
        linhas_escala.append({'coluna': coluna, 'nivel': nivel, 'ordem': ordem})

escala_df = pd.DataFrame(linhas_escala)

# Conferência: todo valor das ordinais na base precisa estar na escala.
# Um nível fora da escala vira ausente no 01 e "item inexistente" no 02.
for coluna, niveis in escala_df.groupby('coluna')['nivel'].apply(list).items():
    valores = df[coluna].dropna()
    if pd.api.types.is_numeric_dtype(valores):
        fora = sorted(set(valores) - set(niveis))
    else:
        fora = sorted(set(valores.astype(str)) - set(map(str, niveis)))
    assert not fora, f'{coluna}: níveis fora da escala {fora}'

# Salvando o arquivo
escala_df.to_csv('../data/processed/escala.csv', index=False, encoding='utf-8', sep=';')
'''


'\nESCALA_NUMERICA_PADRAO = list(range(1, 11))\n\n# Convertendo o dicionário de escalas para o formato longo\nlinhas_escala = []\nfor coluna, niveis in ESCALAS_ORDINAIS.items():\n    if niveis is None:\n        niveis = ESCALA_NUMERICA_PADRAO\n    for ordem, nivel in enumerate(niveis, start=1):\n        linhas_escala.append({\'coluna\': coluna, \'nivel\': nivel, \'ordem\': ordem})\n\nescala_df = pd.DataFrame(linhas_escala)\n\n# Conferência: todo valor das ordinais na base precisa estar na escala.\n# Um nível fora da escala vira ausente no 01 e "item inexistente" no 02.\nfor coluna, niveis in escala_df.groupby(\'coluna\')[\'nivel\'].apply(list).items():\n    valores = df[coluna].dropna()\n    if pd.api.types.is_numeric_dtype(valores):\n        fora = sorted(set(valores) - set(niveis))\n    else:\n        fora = sorted(set(valores.astype(str)) - set(map(str, niveis)))\n    assert not fora, f\'{coluna}: níveis fora da escala {fora}\'\n\n# Salvando o arquivo\nescala_df.to_csv(\'../data/pro